
# Phase 05 — Core Statistical Analysis

## 1. Experiment Overview

### Business Question

Does the redesigned landing page (treatment) produce a statistically significant change in user conversion compared with the existing landing page (control)?

### Experiment Groups

- **Control:** Existing landing page (`old_page`)
- **Treatment:** Redesigned landing page (`new_page`)
- **Primary Metric:** User-level conversion rate
- **Unit of Analysis:** Unique user

### Statistical Design

- **Null Hypothesis (H0):** Treatment and control conversion probabilities are equal.
- **Alternative Hypothesis (H1):** Treatment and control conversion probabilities differ.
- **Statistical Test:** Two-sided two-proportion z-test
- **Significance Level:** 0.05
- **Confidence Level:** 95%
- **Practical Significance Threshold:** +1 percentage point absolute improvement

### Important Data Limitation

The cleaned dataset uses the earliest observed record per user as a provisional assignment proxy.

Original experimental assignments cannot be independently verified because the source dataset does not contain an original assignment log.

Statistical findings must therefore be interpreted with this limitation in mind.


In [15]:

# Step 05.04 — Import statistical analysis libraries

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep
)

print("Phase 05 imports successful.")


Phase 05 imports successful.


In [16]:

# Step 05.05 — Load the cleaned user-level dataset

from pathlib import Path

# Notebook is inside the notebooks/ folder.
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = (
    project_root
    / "data"
    / "processed"
    / "ab_data_provisional_user_level.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Dataset loaded successfully!
Rows: 290,584
Columns: 8


In [17]:

# Step 05.06 — Verify user-level uniqueness

total_rows = len(df)
unique_users = df["user_id"].nunique()
duplicate_users = df["user_id"].duplicated().sum()

print(f"Total rows: {total_rows:,}")
print(f"Unique users: {unique_users:,}")
print(f"Duplicate user IDs: {duplicate_users:,}")
print(f"One row per user: {total_rows == unique_users}")


Total rows: 290,584
Unique users: 290,584
Duplicate user IDs: 0
One row per user: True


In [18]:

# Step 05.07 — Verify experiment group counts

group_counts = df["group"].value_counts()

print("EXPERIMENT GROUP COUNTS")
print("-----------------------")

print(group_counts.to_string())

print("\nTotal users:", f"{group_counts.sum():,}")


EXPERIMENT GROUP COUNTS
-----------------------
group
control      145294
treatment    145290

Total users: 290,584


In [19]:

# Step 05.08 — Verify conversion totals by group

conversion_summary = (
    df.groupby("group")["converted"]
    .agg(
        total_users="count",
        converted_users="sum"
    )
)

conversion_summary["non_converted_users"] = (
    conversion_summary["total_users"]
    - conversion_summary["converted_users"]
)

print("CONVERSION SUMMARY BY GROUP")
print("---------------------------")
print(conversion_summary.to_string())


CONVERSION SUMMARY BY GROUP
---------------------------
           total_users  converted_users  non_converted_users
group                                                       
control         145294            17496               127798
treatment       145290            17265               128025


In [20]:

# Step 05.09 — Calculate conversion rates

conversion_summary["conversion_rate"] = (
    conversion_summary["converted_users"]
    / conversion_summary["total_users"]
)

print("EXPERIMENT CONVERSION RATES")
print("---------------------------")

for group, row in conversion_summary.iterrows():
    print(
        f"{group.capitalize()}: "
        f"{row['conversion_rate']:.4%}"
    )


EXPERIMENT CONVERSION RATES
---------------------------
Control: 12.0418%
Treatment: 11.8831%



## 2. Statistical Hypotheses

### Null Hypothesis (H0)

The underlying conversion probabilities are equal between the treatment and control groups.

$$H_0: p_t = p_c$$

### Alternative Hypothesis (H1)

The underlying conversion probabilities differ between the treatment and control groups.

$$H_1: p_t \ne p_c$$

### Statistical Decision Framework

- Statistical test: Two-sided two-proportion z-test
- Significance level (alpha): 0.05
- If p-value < 0.05: Reject the null hypothesis.
- If p-value >= 0.05: Fail to reject the null hypothesis.

A failure to reject the null hypothesis does not prove that the two landing pages are equivalent.

Statistical conclusions remain conditional on the experiment's assumptions and known assignment-quality limitations.


In [21]:

# Step 05.11 — Prepare inputs for the two-proportion z-test

# Order: Treatment first, Control second

converted_counts = np.array([
    conversion_summary.loc["treatment", "converted_users"],
    conversion_summary.loc["control", "converted_users"]
])

sample_sizes = np.array([
    conversion_summary.loc["treatment", "total_users"],
    conversion_summary.loc["control", "total_users"]
])

print("Z-TEST INPUTS")
print("-------------")
print("Converted users:", converted_counts)
print("Total users:", sample_sizes)


Z-TEST INPUTS
-------------
Converted users: [17265 17496]
Total users: [145290 145294]


In [22]:

# Step 05.12 — Two-sided two-proportion z-test

z_statistic, p_value = proportions_ztest(
    count=converted_counts,
    nobs=sample_sizes,
    alternative="two-sided"
)

print("PRIMARY TWO-PROPORTION Z-TEST")
print("-----------------------------")
print(f"Z-statistic: {z_statistic:.6f}")
print(f"P-value: {p_value:.6f}")
print("Significance level (alpha): 0.05")


PRIMARY TWO-PROPORTION Z-TEST
-----------------------------
Z-statistic: -1.317745
P-value: 0.187589
Significance level (alpha): 0.05


In [23]:

# Step 05.13 — Independently verify the z-test

p_t = converted_counts[0] / sample_sizes[0]
p_c = converted_counts[1] / sample_sizes[1]

pooled_p = converted_counts.sum() / sample_sizes.sum()

pooled_se = np.sqrt(
    pooled_p * (1 - pooled_p)
    * (1 / sample_sizes[0] + 1 / sample_sizes[1])
)

manual_z = (p_t - p_c) / pooled_se
manual_p = 2 * stats.norm.sf(abs(manual_z))

print("MANUAL Z-TEST VERIFICATION")
print("--------------------------")
print(f"Pooled conversion proportion: {pooled_p:.6f}")
print(f"Pooled standard error: {pooled_se:.8f}")
print(f"Manual Z-statistic: {manual_z:.6f}")
print(f"Manual P-value: {manual_p:.6f}")

print("\nVerification:")
print("Z-statistic matches:", np.isclose(manual_z, z_statistic))
print("P-value matches:", np.isclose(manual_p, p_value))


MANUAL Z-TEST VERIFICATION
--------------------------
Pooled conversion proportion: 0.119625
Pooled standard error: 0.00120403
Manual Z-statistic: -1.317745
Manual P-value: 0.187589

Verification:
Z-statistic matches: True
P-value matches: True


In [24]:

# Step 05.14 — Calculate absolute treatment effect

absolute_effect = p_t - p_c

effect_percentage_points = absolute_effect * 100

print("ABSOLUTE TREATMENT EFFECT")
print("-------------------------")

print(f"Control conversion rate: {p_c:.4%}")
print(f"Treatment conversion rate: {p_t:.4%}")

print(
    f"Absolute difference (proportion): "
    f"{absolute_effect:.6f}"
)

print(
    f"Absolute difference (percentage points): "
    f"{effect_percentage_points:+.4f} pp"
)


ABSOLUTE TREATMENT EFFECT
-------------------------
Control conversion rate: 12.0418%
Treatment conversion rate: 11.8831%
Absolute difference (proportion): -0.001587
Absolute difference (percentage points): -0.1587 pp


In [25]:

# Step 05.15 — Calculate relative conversion-rate difference

relative_effect = (p_t - p_c) / p_c

print("RELATIVE TREATMENT EFFECT")
print("-------------------------")

print(f"Control conversion rate: {p_c:.4%}")
print(f"Treatment conversion rate: {p_t:.4%}")

print(
    f"Relative conversion-rate difference: "
    f"{relative_effect:+.4%}"
)


RELATIVE TREATMENT EFFECT
-------------------------
Control conversion rate: 12.0418%
Treatment conversion rate: 11.8831%
Relative conversion-rate difference: -1.3176%


In [26]:

# Step 05.16 — Calculate 95% confidence interval
# Method: Newcombe score-based interval
# Difference: Treatment minus Control

ci_lower, ci_upper = confint_proportions_2indep(
    count1=int(converted_counts[0]),
    nobs1=int(sample_sizes[0]),
    count2=int(converted_counts[1]),
    nobs2=int(sample_sizes[1]),
    compare="diff",
    method="newcomb",
    alpha=0.05
)

print("95% CONFIDENCE INTERVAL")
print("-----------------------")
print("Method: Newcombe")
print(f"Observed effect: {absolute_effect * 100:+.4f} pp")
print(f"Lower bound: {ci_lower * 100:+.4f} pp")
print(f"Upper bound: {ci_upper * 100:+.4f} pp")


95% CONFIDENCE INTERVAL
-----------------------
Method: Newcombe
Observed effect: -0.1587 pp
Lower bound: -0.3947 pp
Upper bound: +0.0773 pp



## 3. Confidence Interval Interpretation

### Estimated Treatment Effect

The observed absolute conversion-rate difference
(treatment minus control) is **-0.1587 percentage points**.

### 95% Confidence Interval

Using the Newcombe score-based method:

- Lower bound: **-0.3947 percentage points**
- Upper bound: **+0.0773 percentage points**

### Statistical Interpretation

The confidence interval includes zero, indicating that
a zero difference remains compatible with the observed
data at the 95% confidence level.

This is consistent with the two-sided z-test:

- Z-statistic: -1.317745
- P-value: 0.187589
- Significance level: 0.05

Because the p-value exceeds 0.05, we fail to reject
the null hypothesis.

The experiment does not provide sufficient statistical
evidence that the redesigned landing page changed
conversion.

### Important Limitations

A non-significant result does not establish equivalence
between the two landing pages.

The confidence interval describes uncertainty under
the statistical model's assumptions.

Original experimental assignments could not be
independently verified, so causal conclusions remain
limited by the provisional assignment methodology.


In [27]:

# Step 05.18 — Evaluate practical significance

# Approved minimum practical improvement:
# +1 percentage point = 0.01 in proportion units

mpe = 0.01

print("PRACTICAL SIGNIFICANCE ASSESSMENT")
print("---------------------------------")

print(f"Approved MPE: {mpe * 100:+.4f} pp")
print(f"Observed effect: {absolute_effect * 100:+.4f} pp")
print(f"95% CI lower: {ci_lower * 100:+.4f} pp")
print(f"95% CI upper: {ci_upper * 100:+.4f} pp")

print("\nAssessment:")
print(
    "Observed effect meets MPE:",
    absolute_effect >= mpe
)

print(
    "95% CI includes +1 pp:",
    ci_lower <= mpe <= ci_upper
)

print(
    "95% CI entirely below MPE:",
    ci_upper < mpe
)


PRACTICAL SIGNIFICANCE ASSESSMENT
---------------------------------
Approved MPE: +1.0000 pp
Observed effect: -0.1587 pp
95% CI lower: -0.3947 pp
95% CI upper: +0.0773 pp

Assessment:
Observed effect meets MPE: False
95% CI includes +1 pp: False
95% CI entirely below MPE: True



## 4. Statistical and Practical Significance

### Statistical Significance

The approved two-sided two-proportion z-test produced:

- Z-statistic: -1.317745
- P-value: 0.187589
- Significance level: 0.05

Because the p-value exceeds 0.05, we fail to reject
the null hypothesis.

The experiment does not provide sufficient statistical
evidence that the redesigned landing page changed
the underlying user conversion probability.

This does not prove that the two pages are equivalent.

### Practical Significance

The approved minimum practical effect (MPE) is
a +1 percentage-point absolute improvement.

Observed results:

- Absolute effect: -0.1587 percentage points
- Relative effect: -1.3176%
- 95% CI: [-0.3947, +0.0773] percentage points

The observed effect does not meet the approved MPE.

Furthermore, the entire 95% confidence interval
lies below the +1 percentage-point threshold.

Under the statistical model's assumptions, the
interval provides evidence against an improvement
as large as the approved practical threshold.

### Business Interpretation

The redesigned landing page did not demonstrate
a statistically significant conversion change or
the targeted +1 percentage-point improvement.

The observed conversion rate was slightly lower
in the treatment group, but the evidence does not
establish that the redesign is harmful.

These findings alone do not constitute a final
product launch decision.

### Scientific Caution

The analysis uses provisional user-level assignments
derived from the earliest observed record.

Because original experimental assignments cannot
be independently verified, statistical conclusions
remain conditional on assumptions about assignment
validity and data quality.

The results should not be presented as definitive
causal evidence.


In [28]:

# Step 05.20 — Verify normal approximation conditions

print("NORMAL APPROXIMATION CHECK")
print("--------------------------")

for group, n in zip(
    ["Treatment", "Control"],
    sample_sizes
):
    expected_successes = n * pooled_p
    expected_failures = n * (1 - pooled_p)

    passes = (
        expected_successes >= 10
        and expected_failures >= 10
    )

    print(f"\n{group}:")
    print(f"Expected successes: {expected_successes:,.2f}")
    print(f"Expected failures: {expected_failures:,.2f}")
    print(f"Normal approximation check passed: {passes}")


NORMAL APPROXIMATION CHECK
--------------------------

Treatment:
Expected successes: 17,380.26
Expected failures: 127,909.74
Normal approximation check passed: True

Control:
Expected successes: 17,380.74
Expected failures: 127,913.26
Normal approximation check passed: True



## 5. Statistical Assumptions and Limitations

### 5.1 Unit of Analysis

The analysis uses 290,584 unique users, with one
retained observation per user.

This prevents duplicate user IDs from being counted
as separate observations in the primary analysis.

However, one row per user does not independently
establish statistical independence.

### 5.2 Binary Outcome

The conversion variable contains valid binary values:

- 1 = Converted
- 0 = Did not convert

The binary outcome requirement is satisfied.

### 5.3 Normal Approximation

Expected successes and failures exceed 10 in both
experimental groups.

The normal approximation condition for the
two-proportion z-test is satisfied.

### 5.4 Sample Allocation

The observed group allocation is approximately 50/50.

The Phase 04 SRM check produced:

- SRM p-value: 0.994079
- Approved SRM alarm threshold: 0.001
- Result: No SRM alarm

This supports consistency with the planned allocation
but does not independently verify valid randomization.

### 5.5 Provisional Assignment Limitation

The dataset does not provide an independently verified
original experimental assignment log.

For repeated users, the earliest observed record was
retained as a provisional assignment proxy.

Therefore, the analysis cannot establish fully
verified intent-to-treat assignment.

### 5.6 Known Data-Quality Issues

The cleaned dataset retains flags identifying:

- 3,894 users with repeated source records
- 1,895 users with cross-group assignments
- 1,949 retained records with group/page mismatches

These categories may overlap and must not be summed
as mutually exclusive populations.

Flagged users were not silently excluded from the
approved primary analysis.

### 5.7 Independence and Causal Interpretation

The statistical analysis assumes appropriate
independence between user-level observations.

Although each user appears once in the cleaned data,
independence and original randomization cannot be
fully verified from the available source records.

Consequently, statistical conclusions remain
conditional on these assumptions.

The results should not be presented as definitive
causal evidence of the redesigned page's impact.

### 5.8 Dataset Provenance

The experiment dataset comes from a public Kaggle
educational source.

Its provenance and licensing limitations are
documented in the project.

These limitations should be disclosed when
presenting the analysis.
